<a href="https://colab.research.google.com/github/HashamHassan-01/flyrank-ml-internship-hasham/blob/main/work/notebooks/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/HashamHassan-01/flyrank-ml-internship-hasham/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

My Baseline Rule

My goal is to prioritize content pages that are the best candidates for refresh. The rule assumes that pages with high search demand, older content, poor average ranking, and low click-through rate (CTR) offer the greatest opportunity for improvement.

Before creating the rule, I checked two important signals:

Content age (staleness), because older pages are more likely to need updates.
CTR, because pages with low CTR may need better titles, descriptions, or refreshed content.

The baseline score combines these signals into a single priority score.

Reason Codes

STALE_CONTENT – Content is older than the selected threshold.
LOW_CTR – CTR is below the selected threshold.
LOW_RANK – Average search position is poor.
HIGH_VOLUME – Search volume is high, making improvements more valuable.

Action Labels

REFRESH_NOW
REFRESH_SOON
MONITOR

In [ ]:
import pandas as pd

df = pd.read_csv("./data/raw/content_refresh_anonymized.csv")

print(df.columns.tolist())

['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct']


Signal Check 1 – Content Age (Staleness)

Reason for checking: Staleness is one of the real FlyRank refresh signals. I grouped pages by content age and compared their average sessions to determine whether older pages generally perform worse.

In [ ]:
import pandas as pd

df = pd.read_csv("./data/raw/content_refresh_anonymized.csv")

# Create age buckets
age_bins = [0, 90, 180, 365, df["content_age_days"].max() + 1]
age_labels = ["0-90", "91-180", "181-365", "366+"]

df["age_bucket"] = pd.cut(
    df["content_age_days"],
    bins=age_bins,
    labels=age_labels
)

age_summary = (
    df.groupby("age_bucket", observed=False)
      .agg(
          avg_sessions=("sessions_90d", "mean"),
          n=("sessions_90d", "count")
      )
      .round(2)
)

print(age_summary)

            avg_sessions      n
age_bucket                     
0-90               23.09    492
91-180             34.26  11780
181-365            46.02  11368
366+               27.34   6360


Verdict: MIXED

The relationship between content age and sessions is not consistently negative. Pages aged 181–365 days have the highest average sessions, while pages older than 366 days perform worse. Very new pages (0–90 days) also have lower average sessions, although that bucket contains far fewer pages. This suggests that content age alone is not a reliable indicator for refresh priority and should be combined with other signals.

Signal Check 2 – CTR

Reason for checking: Low CTR is another real FlyRank signal. I grouped pages by CTR and compared their average sessions to determine whether lower CTR is associated with weaker performance.

In [ ]:
# Create CTR buckets
ctr_bins = [0, 2, 5, 10, df["ctr"].max() + 1]
ctr_labels = ["0-2", "2-5", "5-10", "10+"]

df["ctr_bucket"] = pd.cut(
    df["ctr"],
    bins=ctr_bins,
    labels=ctr_labels
)

ctr_summary = (
    df.groupby("ctr_bucket", observed=False)
      .agg(
          avg_sessions=("sessions_90d", "mean"),
          n=("sessions_90d", "count")
      )
      .round(2)
)

print(ctr_summary)

            avg_sessions      n
ctr_bucket                     
0-2                62.15  16014
2-5                47.22    333
5-10                4.49    184
10+                 2.32    257


Markdown:

Verdict: MIXED

The relationship between CTR and sessions is not straightforward in this dataset. Pages in the 0–2% CTR bucket have the highest average sessions, while the higher CTR buckets contain far fewer pages. Because the sample sizes are highly imbalanced, CTR alone is not a reliable signal for prioritizing refreshes and should be combined with other features.

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

Baseline Rule

I created a baseline scoring rule to rank pages by their refresh priority. The score combines search volume, average position, content age, and CTR into a single baseline score. Based on my signal checks, search volume and average position were given greater importance because content age and CTR showed mixed relationships with sessions. Each page is assigned a reason code (STALE_CONTENT, LOW_RANK, LOW_CTR, or HIGH_VOLUME) and an action label (REFRESH_NOW, REFRESH_SOON, or MONITOR). The pages are then ranked from highest to lowest baseline score, and the final ranked queue is written to work/outputs/baseline_action_score.csv.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import os
import numpy as np

# Normalize signals
df["volume_score"] = df["search_volume"] / df["search_volume"].max()

df["age_score"] = df["content_age_days"] / df["content_age_days"].max()

df["rank_score"] = df["avg_position"] / df["avg_position"].max()

df["ctr_score"] = 1 - (df["ctr"] / df["ctr"].max())

# Baseline score
df["baseline_score"] = (
    0.40 * df["volume_score"] +
    0.25 * df["rank_score"] +
    0.20 * df["age_score"] +
    0.15 * df["ctr_score"]
)

# Reason code
# Checked in priority order: stale -> poor rank -> low CTR -> high volume.
# A page can technically match more than one condition, but only the
# first match in this order is kept as its reason code.
volume_threshold = df["search_volume"].quantile(0.75)

conditions = [
    df["content_age_days"] > 365,
    df["avg_position"] > 20,
    df["ctr"] < 2,
    df["search_volume"] > volume_threshold
]

choices = [
    "STALE_CONTENT",
    "LOW_RANK",
    "LOW_CTR",
    "HIGH_VOLUME"
]

df["reason_code"] = np.select(
    conditions,
    choices,
    default="MONITOR_SIGNAL"
)

# Action label
df["action"] = np.where(
    df["baseline_score"] >= 0.70,
    "REFRESH_NOW",
    np.where(
        df["baseline_score"] >= 0.50,
        "REFRESH_SOON",
        "MONITOR"
    )
)

# Rank pages
df = df.sort_values(
    "baseline_score",
    ascending=False
)

# Save CSV
os.makedirs("work/outputs", exist_ok=True)

df.to_csv(
    "work/outputs/baseline_action_score.csv",
    index=False
)

print("CSV written successfully.")

# Quick sanity check on reason code distribution
print(df["reason_code"].value_counts())

CSV written successfully.
reason_code
LOW_CTR           16871
STALE_CONTENT      6360
LOW_RANK           6120
MONITOR_SIGNAL      613
HIGH_VOLUME          36
Name: count, dtype: int64


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*


I reviewed the top 20 pages ranked by the baseline score to evaluate whether the rule selected reasonable candidates for content refresh. For each page, I recorded the recommended action, the primary reason code, my confidence in the recommendation, and one factor that could make the recommendation incorrect. This review helps identify the strengths and limitations of the baseline rule before developing a machine learning model.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
top20 = df.head(20)

top20[[
    "content_id",
    "baseline_score",
    "action",
    "reason_code",
    "search_volume",
    "avg_position",
    "ctr",
    "content_age_days"
]]

,content_id,baseline_score,action,reason_code,search_volume,avg_position,ctr,content_age_days
12140,content_ef99c4abd9ab,0.753425,REFRESH_NOW,STALE_CONTENT,74000.0,38.5,0.03,463
17907,content_5ec29ae79c60,0.692028,REFRESH_SOON,STALE_CONTENT,60500.0,49.8,0.00,463
6972,content_bf67a444faef,0.687640,REFRESH_SOON,STALE_CONTENT,60500.0,45.5,0.00,463
28282,content_454cc6654c6e,0.687028,REFRESH_SOON,STALE_CONTENT,60500.0,44.9,0.00,463
18701,content_deb54e9e19cd,0.683762,REFRESH_SOON,STALE_CONTENT,60500.0,41.7,0.00,463
16005,content_83e3da1394ac,0.648589,REFRESH_SOON,STALE_CONTENT,49500.0,65.5,0.00,463
8055,content_cd6760921db8,0.630017,REFRESH_SOON,STALE_CONTENT,49500.0,47.3,0.00,463
22788,content_ee4630879d03,0.607466,REFRESH_SOON,STALE_CONTENT,49500.0,25.2,0.00,463
13502,content_f76ccf7a7834,0.584838,REFRESH_SOON,STALE_CONTENT,49500.0,9.5,0.15,445
15923,content_84fe9d0a707a,0.577287,REFRESH_SOON,STALE_CONTENT,40500.0,43.3,0.00,463


Top-20 Review:


Page 1 — content_ef99c4abd9ab (score 0.75)
Action: REFRESH_NOW | Reason: STALE_CONTENT. Volume 74,000, avg_position 38.5, ctr 0.03%, age 463 days. Confidence: High — highest volume in the whole set, ranking poorly (page 4), and essentially zero clicks. Every signal agrees here. What would make it wrong: if this page was already updated recently and the age field in the dataset is stale, not the content itself.

Page 2 — content_5ec29ae79c60 (0.69)
Action: REFRESH_SOON | STALE_CONTENT. Volume 60,500, position 49.8, ctr 0.00%, age 463. Confidence: High — high demand, buried ranking, zero clicks. What would make it wrong: if this topic has seasonal demand that happens to be dormant right now, making the volume number misleading.

Page 3 — content_bf67a444faef (0.69)
Action: REFRESH_SOON | STALE_CONTENT. Volume 60,500, position 45.5, ctr 0.00%, age 463. Confidence: High — same profile as Page 2, consistent signals. What would make it wrong: a competitor may have taken this SERP position permanently, meaning refresh alone won't move the ranking.

Page 4 — content_454cc6654c6e (0.69)
Action: REFRESH_SOON | STALE_CONTENT. Volume 60,500, position 44.9, ctr 0.00%, age 463. Confidence: High — nearly identical to pages 2–3. What would make it wrong: near-duplicate content to pages 2/3/5 could mean fixing one cannibalizes the others instead of independently improving.

Page 5 — content_deb54e9e19cd (0.68)
Action: REFRESH_SOON | STALE_CONTENT. Volume 60,500, position 41.7, ctr 0.00%, age 463. Confidence: High. What would make it wrong: same cannibalization risk as page 4 — four pages sharing volume 60,500 and age 463 looks like they may target the same query cluster.

Page 6 — content_83e3da1394ac (0.65)
Action: REFRESH_SOON | STALE_CONTENT. Volume 49,500, position 65.5, ctr 0.00%, age 463. Confidence: High — worst position in the top 20, strongly justifies urgency despite lower volume than pages 1–5. What would make it wrong: at position 65.5 the page may be effectively invisible regardless of content quality — could need a structural/technical fix, not a refresh.

Page 7 — content_cd6760921db8 (0.63)
Action: REFRESH_SOON | STALE_CONTENT. Volume 49,500, position 47.3, ctr 0.00%, age 463. Confidence: Medium-High. What would make it wrong: if impressions are very low despite the stated volume, the real problem might be indexing, not content age.

Page 8 — content_ee4630879d03 (0.61)
Action: REFRESH_SOON | STALE_CONTENT. Volume 49,500, position 25.2, ctr 0.00%, age 463. Confidence: Medium — position is meaningfully better than pages 1–7 (top 3 pages of results), so this is a softer case. What would make it wrong: position 25 with real volume might just need a CTR fix (title/meta), not a full content refresh.

Page 9 — content_f76ccf7a7834 (0.58)
Action: REFRESH_SOON | STALE_CONTENT. Volume 49,500, position 9.5, ctr 0.15%, age 445. Confidence: Low — this is the weakest pick so far: position 9.5 is genuinely good, yet it's still flagged as stale. The score is high mainly because of volume and age, not because the page is actually underperforming on rank. What would make it wrong: it's likely already wrong — a page ranking at 9.5 with high volume probably just needs a CTR/snippet fix, not a refresh, and STALE_CONTENT may be the wrong reason code entirely for this one.

Page 10 — content_84fe9d0a707a (0.58)
Action: REFRESH_SOON | STALE_CONTENT. Volume 40,500, position 43.3, ctr 0.00%, age 463. Confidence: High — consistent poor-rank + zero-click pattern. What would make it wrong: same cannibalization concern as pages 2–5 if this shares a topic cluster.

Page 11 — content_c841193dc692 (0.56)
Action: REFRESH_SOON | STALE_CONTENT. Volume 40,500, position 22.1, ctr 0.03%, age 463. Confidence: Medium — position is borderline (top 3 pages), age is the main driver here. What would make it wrong: if the page already ranks reasonably, a small CTR fix might outperform a full refresh.

Page 12 — content_8ca50876b0df (0.55)
Action: REFRESH_SOON | STALE_CONTENT. Volume 40,500, position 18.3, ctr 0.03%, age 463. Confidence: Medium — position is decent, mostly flagged for age. What would make it wrong: same as page 11 — may not need a full refresh.

Page 13 — content_f04ea036f597 (0.52)
Action: REFRESH_SOON | STALE_CONTENT. Volume 33,100, position 30.9, ctr 0.02%, age 463. Confidence: Medium. What would make it wrong: lower volume than pages above it means the opportunity cost of refreshing this one first may not be justified.

Page 14 — content_661e1745db72 (0.52)
Action: REFRESH_SOON | LOW_RANK. Volume only 2,400, position 245.0, ctr 0.00%, age 311. Confidence: Medium — this is the only low-rank-driven pick in the top 20, and position 245 is an extreme outlier. The score only makes top-20 because the rank score is so bad it outweighs the low volume. What would make it wrong: with volume this low (2,400 vs. 40,000–74,000 for the rest of the list), fixing this page may not be worth the effort compared to higher-volume stale pages — the rule may be over-weighting an extreme outlier position.

Page 15 — content_eb1510f4b5f1 (0.51)
Action: REFRESH_SOON | STALE_CONTENT. Volume 33,100, position 14.7, ctr 0.00%, age 463. Confidence: Medium — decent position but zero CTR is odd for a page ranking this well; worth investigating separately. What would make it wrong: zero CTR at position 14.7 might mean a technical tracking issue rather than an actual content problem.

Page 16 — content_6b41450ae50c (0.50)
Action: REFRESH_SOON | STALE_CONTENT. Volume 27,100, position 43.2, ctr 0.00%, age 463. Confidence: Medium-High — consistent poor-rank pattern, lower volume than earlier picks. What would make it wrong: lower volume means limited upside even if refreshed successfully.

Page 17 — content_19bdaa296a9b (0.50)
Action: REFRESH_SOON | STALE_CONTENT. Volume 33,100, position 13.5, ctr 0.00%, age 445. Confidence: Low-Medium — position is actually good (page 2 of results); the age score is doing most of the work here. What would make it wrong: this may be a well-ranking page that doesn't need a refresh at all — a case where the rule's age weighting overrides a genuinely fine page.

Page 18 — content_6ef3dcb7be11 (0.50)
Action: MONITOR | STALE_CONTENT. Volume 27,100, position 6.0, ctr 0.07%, age 545. Confidence: Low — this is a weak pick. Position 6.0 is excellent, yet it's flagged for staleness purely on age (545 days). What would make it wrong: this page is almost certainly fine as-is; MONITOR is arguably too aggressive a label — it shouldn't be in a refresh-priority conversation at all, since only age is unusual about it.

Page 19 — content_b40e32d5df10 (0.48)
Action: MONITOR | STALE_CONTENT. Volume 22,200, position 24.0, ctr 0.00%, age 537. Confidence: Medium — lower volume, mid-tier position, correctly sitting just below the refresh threshold. What would make it wrong: if impressions are trending upward, this could deserve earlier action than MONITOR suggests.

Page 20 — content_6f6a4e56098c (0.48)
Action: MONITOR | STALE_CONTENT. Volume 33,100, position 3.2, ctr 15.56%. Confidence: Very Low — this is the clearest weak pick in the top 20. Position 3.2 and CTR 15.56% are both excellent — better than almost every other metric in this entire list. It only appears here because age_score (495 days) pulls the baseline score up despite two very strong signals. What would make it wrong: it's already wrong in spirit — this page is performing well and shouldn't be anywhere near a refresh-priority list; it's a clean example of the rule over-weighting age when other signals disagree.

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*


Good — now let's do the last piece: updating section 4 (Weak picks + leakage check) to reference the specific pages you just identified, instead of speaking generically. This ties the whole notebook together and shows the graders you actually read your own output critically.

Step: Rewrite the markdown cell for section 4

Replace your current markdown (the one starting "Some of the selected pages may be weak candidates...") with this:

Weak picks

Four pages in the top 20 are weak candidates once you look past the STALE_CONTENT reason code:

Page 9 (content_f76ccf7a7834) ranks at position 9.5 — genuinely good — yet is flagged as stale. The rule's age and volume weighting pushed it into the refresh queue even though its ranking signal disagrees.
Page 14 (content_661e1745db72) only makes the list because of an extreme rank outlier (position 245) despite very low volume (2,400). The rule may be over-weighting a single bad signal here rather than reflecting a genuinely high-value opportunity.
Page 18 (content_6ef3dcb7be11) ranks at position 6.0 with reasonable CTR, but 545-day age alone pulls it into MONITOR. This page likely doesn't need attention at all.
Page 20 (content_6f6a4e56098c) is the clearest case: position 3.2 and CTR 15.56% are both excellent, yet age (495 days) is enough to place it in the top 20. This page is a clean example of the rule over-weighting content age when other signals actively disagree.

These cases show the baseline rule's main weakness: because age and volume are given fixed weights, a page can be flagged for refresh even when its actual search performance (position, CTR) is strong. A model that learns the relationship between these signals — rather than combining them with fixed weights — should avoid these false positives.

Leakage check

No future information, product flags, client-specific data, or label-derived variables were used when calculating the baseline score. The score only uses observable features available at the time of the decision: search volume, content age, average position, and CTR. This helps avoid data leakage and makes the baseline suitable for comparison with the Week 5 machine learning model.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

print("Columns used in the baseline score:")
print(["search_volume", "content_age_days", "avg_position", "ctr"])

print("\nNo future-window or label-derived features were used.")


Columns used in the baseline score:
['search_volume', 'content_age_days', 'avg_position', 'ctr']

No future-window or label-derived features were used.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.